In [ ]:
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet50
from torchvision.transforms import transforms
from PIL import Image
from tqdm.notebook import tqdm
import torch
from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

transformTrain = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomGrayscale(),
    transforms.ColorJitter(0.05,0.07,0.01,0.06),
    transforms.RandomInvert(),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406],std = [0.229, 0.224, 0.225])
])
transformTest = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406],std = [0.229, 0.224, 0.225])
])

class LoadingSo(Dataset):
    def __init__(self,df,transforms,is_train):
        self.df = df
        self.transforms = transforms
        self.is_train = is_train
    def __len__(self):
        return len(self.df)
    def __getitem__(self,idx):
        img = self.transforms(Image.open(self.df['image_path'].iloc[idx]).convert('RGB'))
        if self.is_train:
            return img, self.df['number'].iloc[idx]
        else:
            return img

trd = LoadingSo(train_df,transformTrain, True)
tsd = LoadingSo(test_df, transformTest, False)

train = DataLoader(trd,4,True)
test = DataLoader(tsd,1,False)

In [ ]:
model = resnet50(weights='DEFAULT')
model.fc = torch.nn.Identity()

In [ ]:
model.eval()

test_embeddings = []
train_embeddings = []
train_labels = []

with torch.no_grad():
    b = tqdm(test)
    for i in b:
        test_embeddings.append(model(i).detach().cpu().numpy())
    c = tqdm(train)
    for i,j in c:
        train_embeddings.extend(model(i).detach().cpu().numpy())
        train_labels.extend(j)

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
sca = StandardScaler()
cat = CatBoostRegressor(iterations = 100, learning_rate = 0.2, max_depth = 6, rsm = 0.8, random_state = 42, l2_leaf_reg = 0.2)
cat.fit(sca.fit_transform(train_embeddings), train_labels)

0:	learn: 1.5535080	total: 562ms	remaining: 55.6s
1:	learn: 1.3581488	total: 1.14s	remaining: 55.6s
2:	learn: 1.2183786	total: 1.65s	remaining: 53.2s
3:	learn: 1.0912992	total: 2.22s	remaining: 53.3s
4:	learn: 0.9623381	total: 2.81s	remaining: 53.5s
5:	learn: 0.8664145	total: 3.4s	remaining: 53.2s
6:	learn: 0.7807635	total: 4s	remaining: 53.2s
7:	learn: 0.7095845	total: 4.53s	remaining: 52.1s
8:	learn: 0.6362170	total: 5.1s	remaining: 51.6s
9:	learn: 0.5889913	total: 5.7s	remaining: 51.3s
10:	learn: 0.5358886	total: 6.29s	remaining: 50.9s
11:	learn: 0.4996052	total: 6.84s	remaining: 50.1s
12:	learn: 0.4603788	total: 7.42s	remaining: 49.6s
13:	learn: 0.4261852	total: 7.99s	remaining: 49.1s
14:	learn: 0.3966124	total: 8.56s	remaining: 48.5s
15:	learn: 0.3703978	total: 9.14s	remaining: 48s
16:	learn: 0.3411801	total: 9.73s	remaining: 47.5s
17:	learn: 0.3165610	total: 10.3s	remaining: 47s
18:	learn: 0.2932357	total: 10.8s	remaining: 46.2s
19:	learn: 0.2731040	total: 11.3s	remaining: 45.4s


CatBoostRegressor(iterations=100, l2_leaf_reg=0.2, learning_rate=0.2, loss_function='RMSE', max_depth=6, random_state=42, rsm=0.8)

In [ ]:
answer = cat.predict(sca.transform(np.array(test_embeddings).squeeze(1)))
pd.DataFrame({
    'image_path':test_df['image_path'].tolist(),
    'number':answer
}).to_csv('submission.csv',index=False)